In [0]:

df = spark.sql("FROM supply_chain_live2.bronze.raw_supply_chain")
df.display()

In [0]:

metadata = spark.sql("FROM supply_chain_live2.bronze.metadata")
metadata.display()

In [0]:
print(df.columns)

In [0]:
df.select("Customer Email", "Customer Country", "Benefit per order").limit(3).display()

In [0]:
# to_timestamp is important

In [0]:
df.select("Product Description").distinct().display()

In [0]:
# K: Nice to do: 
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

The ABOVE code counts null values in each column of a Spark DataFrame.
Breaking it down piece by piece:
col(column).isNull() — for each value in a column, returns True if it's null, False otherwise.
.cast("int") — converts True/False → 1/0, so nulls become countable numbers.
spark_sum(...) — adds up all those 1s and 0s, giving the total null count per column.
The list comprehension [... for column in df.columns] — repeats this for every column.
.select(...) — applies all those expressions at once, returning a new DataFrame where each column contains its null count.
Example:
nameagecityAlicenullParisnull30null
Result: name=1, age=1, city=1Sonnet 4.6

In [0]:
import re 

#   Customer Email -> customer_email
def to_snake_case(name):
    return re.sub(r"[\s]+", '_', name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.column]
    return  df.toDF(*new_columns)


In [0]:

import re 

#     Customer   Email    -> customer_email
def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case(" Customer    Email  AcCount ")

In [0]:
df_cleaned_columns = rename_columns_to_snake_case(df)
df_cleaned_columns.display()

In [0]:
# K: good to call the functions from utils

More cleaning

In [0]:
df_cleaned_columns.select("shipping_date_(dateorders)").limit(1).display()

In [0]:
# To clean 1 column

from pyspark.sql.functions import to_timestamp, col

# 6/19/2017 4:41
df_cleaned_columns.withColumn(
    "shipping_date", to_timestamp("shipping_date_(dateorders)","M/d/yyyy H:mm")
).select("shipping_date").display()

# a function to  get rid of the () in the headers would be great

In [0]:
# to clean several columns 
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when

# 6/19/2017 4:41
(
    df_cleaned_columns.withColumn(
    "shipping_date", to_timestamp("shipping_date_(dateorders)","M/d/yyyy H:mm")
    )
    .withColumn(
        "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown")),
    )
    .withColumn(
    "customer_country",
    when(col("customer_country") == "EE. UU", "United States").otherwise(col("customer_country"))
    )
    .withColumn(
        "order_date",
        to_timestamp("order_date_(dateorders)","M/d/yyyy H:mm")
    )

).select("shipping_date","shipping_date_(dateorders)","order_zipcode", "customer_country","customer_zipcode").display()


In [0]:
# From the above code, we create a df_cleaned to make it easier
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when

df_cleaned = (
    df_cleaned_columns.withColumn(
        "shipping_date", to_timestamp("shipping_date_(dateorders)", "M/d/yyyy H:mm")
    )
    .withColumn(
        "order_zipcode",
        coalesce(col("order_zipcode").cast("string"), lit("unknown")),
    )
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
    )
    .withColumn(
        "customer_country",
        when(col("customer_country") == "EE. UU.", "United States").otherwise(
            col("customer_country")
        ),
    )
    .withColumn(
        "order_date",
        to_timestamp("order_date_(dateorders)", "M/d/yyyy H:mm"),
    )
).drop(
    "customer_email",
    "customer_password",
    "product_description",
    "shipping_date_(dateorders)",
    "order_date_(dateorders)",
)

df_cleaned.select("customer_country", "order_date", "shipping_date").display()

In [0]:
# K correction of rounding 
# order_item_total, rounding all floats
df_cleaned.select("order_item_quantity", "order_item_product_price", "order_item_discount_rate",
"order_item_total", "sales").display()

# "order_item_total" is a derived column, calculated based on other columns
# order_item_total = quantity*product_price*

In [0]:
# Check again how to round with spark.round from K's code